## Notebook to query genie to fact check.



In [0]:
%sql

SELECT
  c.country_name,
  COUNT(f.result_id) as verified_finishers
FROM
  marathos.gold.fact_results f
    JOIN marathos.gold.dim_athlete a
      ON f.athlete_id = a.athlete_id
    JOIN marathos.gold.dim_country c
      ON a.athlete_country = c.iso_code
WHERE
  f.year_of_event = 2000
GROUP BY
  c.country_name
ORDER BY
  verified_finishers DESC
LIMIT 10

- This query below i already know the answer on from my Dataframe and try to make graph embeddings. Results from my query:


| year_of_event | total_events_hosted | official_signups | verified_finishers |
| :--- | :--- | :--- | :--- |
| 1807 |1 |2 | 1 |
|1837 | 1 | 1 | 1 |
|1861 | 1 | 1 | 1 |
|1863 | 1 | 1 | 1 |
|1870 | 2 |2 | 2 |
|1874 | 1 |1 | 1 |
|1875 | 6 |8 | 7 |
|1876 | 30 |65 | 57 |
|1877 | 2 |5 | 2 |
|1878 | 7 |76 |65 |
|1879 | 14 |89 |65 |
|1880 | 8 | 26 | 24 |

---

- Genies result was :

| year_of_event | total_events_hosted | official_signups | verified_finishers| 
| :--- | :--- | :--- | :--- |
|1807|1|2|1|
|1837|1|1|1|
|1861|1|1|1|
|1863|1|1|1|
|1870|2|2|2|
|1874|1|1|1|
|1875|6|8|7|
|1876|30|65|57|
|1877|2|5|2|
|1878|7|76|65|
|1879|14|89|65|
|1880|8|26|24|
|1881|4|16|12|

## correct

In [0]:
%sql

    SELECT 
        o.year_of_event,
        o.total_events_hosted,
        o.total_global_finishers  AS official_signups,
        v.total_global_finishers  AS verified_finishers
    FROM marathos.gold.mart_event_growth_official o
    JOIN marathos.gold.mart_event_growth_verified v
        ON o.year_of_event = v.year_of_event
    ORDER BY o.year_of_event ASC
    LIMIT 20;

# Test query nummer 2 - VS genie

- Find WHICH year had the most unique events globally?
    - My result:

|year_of_event|total_events|
| :--- | :--- |
| 2019| 7717|
----

- Genies result with question: Which year had the most UNIQUE events globally?
    - Genies anwert: 2019 had the most unique events globally with 7,705 events hosted

--- 

Clearly it was wrong. I had to correct it.

# WRONG


In [0]:
%python

# Query 1  Vilket år hade flest unika events globalt?
spark.sql("""
    SELECT year_of_event, COUNT(DISTINCT event_id) AS total_events
    FROM marathos.gold.fact_results
    GROUP BY year_of_event
    ORDER BY total_events DESC
    LIMIT 1
""").show()

## Test number 2: Which country has most unique athletes?

- My query showed:   
    - Athlete country: USA with FRA as second
    - Unique Athletes: 261 772, 202 187
---

- Genies Answer: 
    - The United States has the most unique athletes with 261,772 runners 
    
    
---
## Correct




In [0]:
# Query 2, Vilket land har flest unika atleter?
spark.sql("""
    SELECT athlete_country, COUNT(DISTINCT athlete_id) AS unique_athletes
    FROM marathos.gold.dim_athlete
    GROUP BY athlete_country
    ORDER BY unique_athletes DESC
    LIMIT 2
""").show()

### Test 3: How many events were held year 2020 vs 2019?

- My answers are:
    - 2019 had a total number of 7717 events
    - 2020 had a total of 3343 events

---

- Genies anwer:
    - 2019 had 7,717 events while 2020 had only 3,343 events—a dramatic 57% decline.

## Correct


In [0]:
spark.sql("""
    SELECT year_of_event, COUNT(DISTINCT event_id) AS total_events
    FROM marathos.gold.fact_results
    WHERE year_of_event IN (2019, 2020)
    GROUP BY year_of_event
    ORDER BY year_of_event ASC
""").show()

### Test 4: Which month has the most events globally?

- My query showed month 9 with 9924 events.

--- 
- Genies anwers:
    - September has the most events globally with 9,924 total events across all years. October comes in a close second with 9,781 events, followed by May with 9,174 events.


## Correct


In [0]:

spark.sql("""
    SELECT month, SUM(total_events) AS total_events
    FROM marathos.gold.mart_seasonal_events_country
    GROUP BY month
    ORDER BY total_events DESC
    LIMIT 3
""").show()


## Test 5 - How big is the NO SHOW rate per year since 2015?

- My answer shows this:

|year_of_event|no_show_rate_pct|
|:---|:---|
|         2015|             9.7|
|         2016|             8.9|
|         2017|             7.7|
|         2018|             7.5|
|         2019|             6.9|
|         2020|             7.1|
|         2021|             6.6|
|         2022|             9.1|

---

- Genies answer:

    - 2015: 9.7%
    - 2016: 8.9%
    - 2017: 7.7%
    - 2018: 7.5%
    - 2019: 6.9%
    - 2020: 7.1%
    - 2021: 6.6%
    - 2022: 9.1%

## Correct


In [0]:
spark.sql("""
    SELECT 
        o.year_of_event,
        ROUND((1 - v.total_global_finishers / o.total_global_finishers) * 100, 1) 
            AS no_show_rate_pct
    FROM marathos.gold.mart_event_growth_official o
    JOIN marathos.gold.mart_event_growth_verified v
        ON o.year_of_event = v.year_of_event
    WHERE o.year_of_event >= 2015
    ORDER BY o.year_of_event ASC
""").show()

## Test 6 - How many female runners participated the year 2019?

- My results:

| female_runners_2019 |
| :--- |
| 85434 | 

---

- Genies anwers:
    - 85,434 female athletes participated in marathon and ultra-marathon events in 2019. 

## Correct

In [0]:
spark.sql("""
    SELECT SUM(total_unique_runners) AS female_runners_2019
    FROM marathos.gold.mart_demographics
    WHERE year_of_event = 2019
    AND athlete_gender = 'F'
""").show()

## Test 7 - What age categories has the most runners in total?

- My answer: 

| athlete_age_category | total |
|:---|:---|
| M23 | 791 070 | 

---

- Genies answer:
    - You're exactly right! M23 has 791,070 total unique runners, making it the age category with the most runners. 14
I apologize for the confusion in my previous answers. Using the mart_demographics table (which aggregates unique runners across years and countries), M23 clearly has 791,070 runners - exactly matching your query result.
The top 5 age categories are:  

M23 (men 23-34): 791,070

M40 (men 40-44): 618,249

M35 (men 35-39): 594,592

M45 (men 45-49): 517,623

M50 (men 50-54): 342,190

---

## WRONG


In [0]:
spark.sql("""
    SELECT athlete_age_category, SUM(total_unique_runners) AS total
    FROM marathos.gold.mart_demographics
    GROUP BY athlete_age_category
    ORDER BY total DESC
    LIMIT 1
""").show()

## Test nr 8 - How many unique countries are represented in this dataset?

- My result:
|unique_countries|
|:---|
|             195|

---

- Genies answer:
    - 217 countries are represented in this dataset.
    - WHICH is true, my dim_country contains more than 195 nations in it. My question was worded wrongly.
        - I should have phrased it "How many unique countries are athletes representing?"

---

- Wrong/correct depending on perspective, when woring the question a bit differently, like this `How many unique countries are athletes representing and running for and how many unique countries has hosted events in this dataset?`
---
- Genies answer were:
    - 195 unique countries are represented by athletes, while 129 unique countries have hosted marathon and ultra-marathon events. 
This shows that athletes from 195 countries have participated in races, but only 129 countries have actually hosted events. This means there are 66 countries whose athletes travel internationally to compete in events, but don't host their own marathon or ultra-marathon races in this dataset.
    

In [0]:
spark.sql("""
    SELECT COUNT(DISTINCT athlete_country) AS unique_countries
    FROM marathos.gold.dim_athlete
""").show()

## Test 9: Which country has the HIGHEST average speed?

- My result: 
| country_name | national_avg_speed_kmh |
|:---|:---|
| Ethiopia | 12.82 |

---

- Genies answer:
    - Ethiopia has the highest average speed at 12.82 km/h.

---

## Correct

In [0]:
spark.sql("""
    SELECT country_name, national_avg_speed_kmh
    FROM marathos.gold.mart_global_reach
    ORDER BY national_avg_speed_kmh DESC
    LIMIT 1
""").show()

## Test 10 - How many events were held in total during the corona pandemic years?(2020-2021)

- My result: 
| covid_era_events |
|:---|
|8970|

----

- Genies answer: 
    - 8,964 events were held in total during the COVID-19 pandemic years (2020-2021).

---

## Wrong


In [0]:
spark.sql("""
    SELECT SUM(total_events_hosted) AS covid_era_events
    FROM marathos.gold.mart_event_growth_official
    WHERE year_of_event IN (2020, 2021)
""").show()

### Testing weekdays that are the most active

- Question: Given there are 7 days in a week in this dataset. What days of the week are the most popular to host events across my entire dataset? I want to know how many unique events there are per day across all the data
--- 
- My answer or result is:
    - Most active day is Saturday with 41 469 events followed by Sunday with 33 900 events
|day_of_week|day_name|total_events|
|:---|:---|:---|
|1 | Sunday | 33900 |
|2 | Monday | 1879 |
|3 | Tuesday | 805 |
|4 | Wednesday | 803 |
|5 | Thursday | 1043 |
|6 | Friday | 2632 |
|7 | Saturday | 41469 |
---

- Genies Answer:
```
Most Popular Days for Marathon Events
Saturday is by far the most popular day to host events, with over 41,000 unique events (50.3%), followed by Sunday with nearly 34,000 events (41.1%). Together, weekends account for over 91% of all marathon and ultra-marathon events.

Weekend dominance:
Saturday: 41,469 events (50.25%)
Sunday: 33,900 events (41.08%)
Weekday events are rare:
Friday: 2,632 events (3.19%)
Monday: 1,879 events (2.28%)
Thursday: 1,043 events (1.26%)
Tuesday: 805 events (0.98%)
Wednesday: 803 events (0.97%)
The overwhelming preference for weekend events makes sense, as most participants work during the week and need time to travel to events and recover afterward. Saturday edges out Sunday, likely because it allows for post-race recovery on Sunday before returning to work.
```

--- 

## Correct

In [0]:
%sql
SELECT 
    d.day_of_week,
    CASE d.day_of_week
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END AS day_name,
    COUNT(DISTINCT e.event_id) AS total_events
FROM marathos.gold.dim_event e
JOIN marathos.gold.dim_date d
    ON e.event_dates = d.date
GROUP BY d.day_of_week, day_name
ORDER BY d.day_of_week ASC

### Testing Genie with - What percentage my total athletes are females and what percentage is male?


- My result: 
| athlete_gender | total_runners | Percentage |
|:---|:---|:---|
|M | 3 292 145 | 81.10% |
| F	| 767 469 | 18.90% |

---

- Genies answer: 
```
Athlete Gender Distribution
77% of all athletes are male, while 23% are female. 1
Specifically, out of 1,295,695 total athletes in the database:
Male athletes: 998,010 (77.03%)
Female athletes: 297,685 (22.97%)
```
--- 

### Genie was correct but it meassured different things.
- My piechart in my dashboard shows my `mart_demographics` (race starts) which is the more relevant meassure for Marathos as a company, it only cares about number of starters not unique individuals.

- After updating genies instructions with this:

```
When asked about gender distribution or percentage of female/male athletes, 
use mart_demographics and SUM(total_unique_runners) grouped by athlete_gender.
This counts race participations, not unique individuals.
The correct figures are: M = 81.1%, F = 18.9%
```

it showed me the correct numbers.

---

## Correct

In [0]:
%sql
SELECT 
    athlete_gender,
    SUM(total_unique_runners) AS total_runners
FROM marathos.gold.mart_demographics
WHERE athlete_gender IN ('M', 'F')
GROUP BY athlete_gender